# Random rollout: episodes inside a task

## Point

The [previous notebook](00_episodes_to_tasks.ipynb) showed why memory across attempts matters. Now trace the boundaries that make those attempts one task, using the usual Gymnasium `reset` and `step` loop.

## Objective

This example shows the reuse argument in practice: pass an existing CartPole environment to `RepeatTaskEnv`, keep its observations, actions, and rewards, and obtain a succession of episodes with a task boundary. The same composition works with other compatible Gymnasium environments.

Learn the inputs you pass in and the values that come back, and tell an inner episode ending from the task ending. A random CartPole policy is only here so each call has a concrete value. The episodes are three steps long, and each task contains two of them. The random policy does not adapt; it lets us inspect the protocol an adapting agent would use. The cell prints the spaces, then every return.

## Inputs

The wrapper takes an existing Gymnasium environment. You still call `reset` and `step`. The action space is the inner environment's action space.

| Input | Role |
| --- | --- |
| `env` | Inner environment. An episode limit on this env limits each inner episode. |
| `max_task_episodes` | How many completed inner episodes end the task. |
| `reset()` or `reset(seed=None)` | Draws the next episode seed and starts a new task (default `per_episode` mode). |
| `reset(seed=...)` with an integer | Reseeds the wrapper RNG, draws an episode seed, and starts a new task. |
| `action_space.seed(...)` | Seeds action sampling separately from the wrapper RNG. |
| `step(action)` | One action sampled from `action_space`. |

## Outputs

`reset` returns `(observation, info)`. `step` returns `(observation, reward, terminated, truncated, info)`.

`observation` is always a dictionary:

| Field | Meaning |
| --- | --- |
| `observation` | The inner observation, unchanged. For CartPole this is a length-4 vector. |
| `episode_terminated` | Boolean in `Discrete(2)`: whether the inner episode terminated. |
| `episode_truncated` | Boolean in `Discrete(2)`: whether the inner episode truncated. |

`terminated` and `truncated` describe the task. Episode flags mark the end of an attempt. The following `step` ignores its action and returns a reset frame with reward 0 and all flags false. This example ends the task when its two-episode budget is used up.

After either task flag is true, `step` raises `ResetNeeded` until you call `reset`.

At an inner episode boundary, an adapting agent would process the final outcome and retain task knowledge for the next attempt. Calling public `reset()` there would start a new task early. The wrapper resets the environment between attempts; the agent owns its memory. Keep that memory between episodes of the same task and clear it when the task ends.


In [ ]:
import gymnasium as gym

from repeat_task_gym import RepeatTaskEnv


def show(call, observation, reward=None, terminated=None, truncated=None, info=None):
    state = observation["observation"]
    fields = (
        f"observation={state}"
        f" episode_terminated={observation['episode_terminated']}"
        f" episode_truncated={observation['episode_truncated']}"
    )
    if reward is not None:
        fields += f" reward={reward} terminated={terminated} truncated={truncated}"
    if info is not None:
        fields += f" info={info}"
    print(f"{call} -> {fields}")


env = RepeatTaskEnv(
    gym.make("CartPole-v1", max_episode_steps=3), max_task_episodes=2
)

print("Inputs")
print("  env: CartPole-v1, max_episode_steps=3")
print("  max_task_episodes: 2")
print(f"  action_space: {env.action_space}")
print("  reset seed: 0")
print("  action seed: 0")
print()
print("Outputs")
print(f"  observation_space: {env.observation_space}")
print()

try:
    observation, info = env.reset(seed=0)
    env.action_space.seed(0)
    assert observation in env.observation_space
    assert not observation["episode_terminated"] and not observation["episode_truncated"]
    show("reset(seed=0)", observation, info=info)

    expect_reset_frame = False
    for step in range(1, 8):
        action = int(env.action_space.sample())
        observation, reward, terminated, truncated, info = env.step(action)
        assert observation in env.observation_space
        show(
            f"step({action})",
            observation,
            reward,
            terminated,
            truncated,
            info,
        )
        episode_terminated = observation["episode_terminated"]
        episode_truncated = observation["episode_truncated"]
        if expect_reset_frame:
            assert not episode_terminated and not episode_truncated
            assert reward == 0.0
            assert not terminated and not truncated
            expect_reset_frame = False
        elif episode_terminated or episode_truncated:
            assert episode_truncated and not episode_terminated  # TimeLimit ends each short episode.
            assert not terminated
            if step < 7:
                assert not truncated
                expect_reset_frame = True
            else:
                assert truncated
        else:
            assert reward == 1.0 and not terminated and not truncated

    try:
        env.step(env.action_space.sample())
    except gym.error.ResetNeeded as error:
        print(f"step() after the task -> {type(error).__name__}")
    else:
        raise AssertionError("step after the task should require reset()")

    observation, info = env.reset()
    assert observation in env.observation_space
    assert not observation["episode_terminated"] and not observation["episode_truncated"]
    show("reset()", observation, info=info)
finally:
    env.close()


## What the trace tells us

The first `episode_truncated=True` ends an attempt, while both task flags remain false. The next call starts the second attempt through a reset frame. Only the second episode end sets the task's `truncated=True`; public `reset()` then draws the next episode seed and begins a new task. No episode count, progress, or agent memory should continue from the completed task.

The episode signal lets an agent distinguish a fresh attempt from an ordinary transition without discarding what it learned about the task. The task flags tell the outer loop when the entire task is over.

Next: [02 — Task termination](02_task_termination.ipynb) chooses whether success or the attempt budget ends a task.
